# Disc Injection-Moulding Cooling Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tuesdaythe13th/warp/blob/claude/disc-config-struct-ndXr3/notebooks/disc_cooling_sim.ipynb)

A GPU-accelerated physics simulation of disc cooling after injection moulding, using [NVIDIA Warp](https://github.com/NVIDIA/warp).

**What this notebook covers:**
- 2-D axisymmetric heat diffusion in a polymer disc (cylindrical coordinates)
- Avrami-based crystallinity evolution (PET + CSR + PTFE blend)
- Warp-risk scoring: thermal gradient × crystallinity asymmetry → predicted deflection
- Radial plots and 2-D field visualisations

**Physics upgrade path (beyond this notebook):**
1. Replace the placeholder Avrami kinetics with a Nakamura model fitted to DSC data for your actual formulation.
2. Add a thermoelastic plate / eigenstrain solver so `warp_risk` becomes out-of-plane deflection in millimetres.

## 1 — Install dependencies

In [ ]:
!pip install -q warp-lang matplotlib

## 2 — Imports and Warp initialisation

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warp as wp

wp.config.quiet = True
wp.init()

DEVICE = "cuda" if wp.get_cuda_device_count() > 0 else "cpu"
print(f"Running on: {DEVICE}")

## 3 — Warp structs

Two structs are used to keep kernel signatures short:

| Struct | Purpose |
|---|---|
| `DiscConfig` | Geometry and material properties (fixed for a given disc) |
| `CoolingParams` | Process parameters (vary between simulation runs) |

In [ ]:
@wp.struct
class DiscConfig:
    """Geometry and material properties of the disc.

    Attributes:
        radius:    Outer radius (m).
        thickness: Disc thickness (m).
        dr:        Radial grid spacing (m).
        dz:        Axial grid spacing (m).
        k:         Thermal conductivity (W/m·K).
        rho:       Density (kg/m³).
        cp:        Specific heat capacity (J/kg·K).
        T_g:       Glass-transition temperature (K).
        T_m:       Melting temperature (K).
        nx:        Number of radial grid cells.
        nz:        Number of axial grid cells.
    """
    radius:    float
    thickness: float
    dr:        float
    dz:        float
    k:         float
    rho:       float
    cp:        float
    T_g:       float
    T_m:       float
    nx:        int
    nz:        int


@wp.struct
class CoolingParams:
    """Process parameters for one simulation run.

    Attributes:
        T_init:         Initial melt temperature (K).
        T_mold:         Mould wall temperature (K).
        dt:             Time step (s).
        total_time:     Total simulation time (s).
        avrami_k0:      Avrami rate pre-factor (1/s).
        avrami_n:       Avrami exponent (dimensionless).
        chi_max:        Maximum achievable crystallinity (0–1).
        warp_temp_coeff: Weight of thermal gradient in warp-risk score.
        warp_chi_coeff:  Weight of crystallinity asymmetry in warp-risk score.
    """
    T_init:          float
    T_mold:          float
    dt:              float
    total_time:      float
    avrami_k0:       float
    avrami_n:        float
    chi_max:         float
    warp_temp_coeff: float
    warp_chi_coeff:  float

## 4 — Warp kernels

In [ ]:
@wp.func
def idx(i: int, j: int, nz: int) -> int:
    """Row-major flat index for the (nx × nz) grid."""
    return i * nz + j


@wp.func
def clamp_i(x: int, lo: int, hi: int) -> int:
    return wp.min(wp.max(x, lo), hi)


@wp.kernel
def init_temperature(
    T:      wp.array(dtype=wp.float32),
    params: CoolingParams,
):
    tid = wp.tid()
    T[tid] = wp.float32(params.T_init)


@wp.kernel
def init_scalar(
    a:     wp.array(dtype=wp.float32),
    value: float,
):
    tid = wp.tid()
    a[tid] = wp.float32(value)


@wp.kernel
def step_temperature(
    T_in:   wp.array(dtype=wp.float32),
    T_out:  wp.array(dtype=wp.float32),
    config: DiscConfig,
    params: CoolingParams,
):
    """Explicit finite-difference heat diffusion in cylindrical coordinates.

    Solves  ∂T/∂t = α (∂²T/∂r² + (1/r)∂T/∂r + ∂²T/∂z²)
    with Dirichlet mould-wall BCs on the top and bottom (z) faces and
    Neumann (zero-flux) BCs on the axis (r=0) and outer radius.
    """
    tid = wp.tid()
    i = tid // config.nz
    j = tid - i * config.nz

    alpha = config.k / (config.rho * config.cp)
    dr    = config.dr
    dz    = config.dz
    r     = (float(i) + 0.5) * dr

    # Dirichlet BC on top/bottom mould walls.
    if j == 0 or j == config.nz - 1:
        T_out[tid] = wp.float32(params.T_mold)
        return

    # Neumann BC: mirror stencil at axis and outer edge.
    im = clamp_i(i - 1, 0, config.nx - 1)
    ip = clamp_i(i + 1, 0, config.nx - 1)
    if i == 0:
        im = 1
    if i == config.nx - 1:
        ip = config.nx - 2

    jm = j - 1
    jp = j + 1

    Tc  = T_in[idx(i,  j,  config.nz)]
    Trm = T_in[idx(im, j,  config.nz)]
    Trp = T_in[idx(ip, j,  config.nz)]
    Tzm = T_in[idx(i,  jm, config.nz)]
    Tzp = T_in[idx(i,  jp, config.nz)]

    d2Tdr2       = (Trp - 2.0 * Tc + Trm) / (dr * dr)
    dTdr_over_r  = 0.0
    if i > 0:
        dTdr_over_r = (Trp - Trm) / (2.0 * dr * r)
    d2Tdz2 = (Tzp - 2.0 * Tc + Tzm) / (dz * dz)

    lap   = d2Tdr2 + dTdr_over_r + d2Tdz2
    Tnew  = Tc + params.dt * alpha * lap
    T_out[tid] = wp.float32(Tnew)


@wp.kernel
def update_crystallinity(
    T:       wp.array(dtype=wp.float32),
    chi_in:  wp.array(dtype=wp.float32),
    chi_out: wp.array(dtype=wp.float32),
    config:  DiscConfig,
    params:  CoolingParams,
):
    """Simple Avrami-style crystallisation kinetics.

    Crystal growth is fastest mid-way between T_g and T_m and saturates
    at chi_max.  Replace with a Nakamura model for production use.
    """
    tid  = wp.tid()
    temp = T[tid]
    chi  = chi_in[tid]

    if temp > config.T_g and temp < config.T_m:
        x    = (temp - config.T_g) / (config.T_m - config.T_g)
        x    = wp.max(0.0, wp.min(1.0, x))
        rate = params.avrami_k0 * x * (1.0 - chi / params.chi_max)
        chi  = chi + params.dt * params.avrami_n * rate
        chi  = wp.max(0.0, wp.min(params.chi_max, chi))

    chi_out[tid] = wp.float32(chi)


@wp.kernel
def compute_warp_risk(
    T:         wp.array(dtype=wp.float32),
    chi:       wp.array(dtype=wp.float32),
    warp_risk: wp.array(dtype=wp.float32),
    config:    DiscConfig,
    params:    CoolingParams,
):
    """Score each radial position by thermal gradient and crystallinity asymmetry.

    Only threads at the mid-plane (j == nz/2) write to warp_risk[i].
    """
    tid = wp.tid()
    i   = tid // config.nz
    j   = tid - i * config.nz

    if j != config.nz // 2:
        return

    top = T[idx(i, 0,              config.nz)]
    bot = T[idx(i, config.nz - 1,  config.nz)]
    mid = T[idx(i, j,              config.nz)]

    chi_top = chi[idx(i, 1,              config.nz)]
    chi_bot = chi[idx(i, config.nz - 2,  config.nz)]

    dT_thickness = wp.abs(top - bot)
    dT_mid       = wp.abs(mid - 0.5 * (top + bot))
    dchi         = wp.abs(chi_top - chi_bot)

    risk = (
        params.warp_temp_coeff * (dT_thickness + dT_mid)
        + params.warp_chi_coeff * dchi
    )
    warp_risk[i] = wp.float32(risk)

## 5 — Simulation class

In [ ]:
class ArtifexCoolingSim:
    """Manages GPU arrays and orchestrates the time-stepping loop."""

    def __init__(self, config: DiscConfig, device: str = "cpu"):
        self.config = config
        self.device = device
        self.n      = config.nx * config.nz

        self.T_a       = wp.zeros(self.n,      dtype=wp.float32, device=device)
        self.T_b       = wp.zeros(self.n,      dtype=wp.float32, device=device)
        self.chi_a     = wp.zeros(self.n,      dtype=wp.float32, device=device)
        self.chi_b     = wp.zeros(self.n,      dtype=wp.float32, device=device)
        self.warp_risk = wp.zeros(config.nx,   dtype=wp.float32, device=device)

    def simulate_cooling(self, params: CoolingParams) -> dict:
        """Run the full cooling simulation and return result metrics.

        Args:
            params: Process parameters for this run.

        Returns:
            Dictionary with scalar metrics and 2-D numpy field arrays.
        """
        wp.launch(init_temperature, dim=self.n, inputs=[self.T_a, params],    device=self.device)
        wp.launch(init_scalar,      dim=self.n, inputs=[self.chi_a, 0.0],    device=self.device)

        steps = int(params.total_time / params.dt)
        for _ in range(steps):
            wp.launch(
                kernel=step_temperature,
                dim=self.n,
                inputs=[self.T_a, self.T_b, self.config, params],
                device=self.device,
            )
            self.T_a, self.T_b = self.T_b, self.T_a

            wp.launch(
                kernel=update_crystallinity,
                dim=self.n,
                inputs=[self.T_a, self.chi_a, self.chi_b, self.config, params],
                device=self.device,
            )
            self.chi_a, self.chi_b = self.chi_b, self.chi_a

        wp.launch(
            kernel=compute_warp_risk,
            dim=self.n,
            inputs=[self.T_a, self.chi_a, self.warp_risk, self.config, params],
            device=self.device,
        )

        T_np    = self.T_a.numpy().reshape((self.config.nx, self.config.nz))
        chi_np  = self.chi_a.numpy().reshape((self.config.nx, self.config.nz))
        risk_np = self.warp_risk.numpy()

        max_delta_t      = float(np.max(np.abs(T_np[:, 0] - T_np[:, -1])))
        avg_chi_groove   = float(np.mean(chi_np[:, 1]))
        max_warp_risk    = float(np.max(risk_np))
        is_ok = avg_chi_groove < 0.15 and max_warp_risk < 15.0

        return {
            "max_delta_t":       max_delta_t,
            "avg_chi_groove":    avg_chi_groove,
            "max_warp_risk":     max_warp_risk,
            "is_ok":             is_ok,
            "final_T_field":     T_np,
            "final_chi_field":   chi_np,
            "warp_risk_radial":  risk_np,
        }

## 6 — Set up geometry and process parameters

Edit the cells below to match your disc geometry and material properties.
The default values approximate a 120 mm PET optical disc cooled in a 25 °C mould.

In [ ]:
# ---------------------------------------------------------------------------
# Geometry
# ---------------------------------------------------------------------------
RADIUS_MM    = 60.0   # outer radius (mm)
THICKNESS_MM = 1.2    # disc thickness (mm)
NX           = 60     # radial grid cells
NZ           = 20     # axial grid cells (must be ≥ 4)

radius    = RADIUS_MM    * 1e-3
thickness = THICKNESS_MM * 1e-3
dr        = radius    / NX
dz        = thickness / NZ

# ---------------------------------------------------------------------------
# Material (PET + 10 % CSR + 5 % PTFE blend, approximate values)
# ---------------------------------------------------------------------------
config        = DiscConfig()
config.radius    = radius
config.thickness = thickness
config.dr        = dr
config.dz        = dz
config.k         = 0.29      # W/m·K
config.rho       = 1360.0    # kg/m³
config.cp        = 1250.0    # J/kg·K
config.T_g       = 348.0     # K  (~75 °C)
config.T_m       = 533.0     # K  (~260 °C)
config.nx        = NX
config.nz        = NZ

# Fourier stability: dt < dr²/(2α) and dt < dz²/(2α)
alpha   = config.k / (config.rho * config.cp)
dt_max  = 0.4 * min(dr, dz) ** 2 / alpha
print(f"Thermal diffusivity α = {alpha:.2e} m²/s")
print(f"Stability limit dt_max = {dt_max*1e3:.3f} ms  →  using dt = {dt_max*1e3*0.9:.3f} ms")

In [ ]:
# ---------------------------------------------------------------------------
# Process parameters — edit these to explore different moulding conditions
# ---------------------------------------------------------------------------
params                   = CoolingParams()
params.T_init            = 553.0          # K  (melt inlet, ~280 °C)
params.T_mold            = 298.0          # K  (mould wall, ~25 °C)
params.dt                = dt_max * 0.9   # s  (90 % of stability limit)
params.total_time        = 8.0            # s  (total cooling window)
params.avrami_k0         = 0.005          # 1/s  — placeholder; fit from DSC
params.avrami_n          = 2.5            # —     placeholder Avrami exponent
params.chi_max           = 0.35           # max crystallinity for this blend
params.warp_temp_coeff   = 0.2            # K⁻¹  — normalisation weight
params.warp_chi_coeff    = 50.0           # —     crystallinity weight

steps = int(params.total_time / params.dt)
print(f"Time steps: {steps:,}   (dt = {params.dt*1e3:.3f} ms, total = {params.total_time} s)")

## 7 — Run the simulation

In [ ]:
sim     = ArtifexCoolingSim(config, device=DEVICE)
results = sim.simulate_cooling(params)

print("\n=== Results ===")
print(f"  Max top-bottom ΔT         : {results['max_delta_t']:.2f} K")
print(f"  Mean groove crystallinity : {results['avg_chi_groove']:.4f}")
print(f"  Max warp-risk score       : {results['max_warp_risk']:.3f}")
print(f"  Quality gate passed       : {results['is_ok']}")

## 8 — Visualise the results

In [ ]:
T_field   = results["final_T_field"]    # (nx, nz)
chi_field = results["final_chi_field"]  # (nx, nz)
risk      = results["warp_risk_radial"] # (nx,)

r_mm = np.linspace(dr / 2, radius - dr / 2, NX) * 1e3   # mm
z_mm = np.linspace(dz / 2, thickness - dz / 2, NZ) * 1e3  # mm
R, Z = np.meshgrid(r_mm, z_mm, indexing="ij")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ---- Temperature field ----
ax = axes[0]
im = ax.pcolormesh(R, Z, T_field, cmap="inferno", shading="auto")
fig.colorbar(im, ax=ax, label="Temperature (K)")
ax.set_xlabel("Radius (mm)")
ax.set_ylabel("Thickness (mm)")
ax.set_title(f"Final temperature field  (t = {params.total_time} s)")

# ---- Crystallinity field ----
ax = axes[1]
im = ax.pcolormesh(R, Z, chi_field, cmap="viridis", shading="auto",
                   vmin=0, vmax=params.chi_max)
fig.colorbar(im, ax=ax, label="Crystallinity χ")
ax.set_xlabel("Radius (mm)")
ax.set_ylabel("Thickness (mm)")
ax.set_title("Final crystallinity field")

# ---- Warp-risk radial profile ----
ax = axes[2]
ax.plot(r_mm, risk, color="crimson", linewidth=2)
ax.axhline(15.0, color="gray", linestyle="--", linewidth=1, label="Quality threshold")
ax.fill_between(r_mm, risk, 15.0,
                where=(risk > 15.0), color="crimson", alpha=0.2, label="At-risk zone")
ax.set_xlabel("Radius (mm)")
ax.set_ylabel("Warp-risk score")
ax.set_title("Radial warp-risk profile")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 9 — Radial mid-plane profiles

In [ ]:
mid = NZ // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(r_mm, T_field[:, mid] - 273.15, label="Mid-plane", color="tab:orange")
ax.plot(r_mm, T_field[:, 0]  - 273.15, label="Top surface", color="tab:blue", linestyle="--")
ax.plot(r_mm, T_field[:, -1] - 273.15, label="Bottom surface", color="tab:green", linestyle=":")
ax.set_xlabel("Radius (mm)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Radial temperature profiles")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(r_mm, chi_field[:, mid],  label="Mid-plane",     color="tab:purple")
ax.plot(r_mm, chi_field[:, 1],    label="Near top wall", color="tab:blue",  linestyle="--")
ax.plot(r_mm, chi_field[:, -2],   label="Near bot wall", color="tab:green", linestyle=":")
ax.axhline(0.15, color="gray", linestyle="--", linewidth=1, label="χ threshold (0.15)")
ax.set_xlabel("Radius (mm)")
ax.set_ylabel("Crystallinity χ")
ax.set_title("Radial crystallinity profiles")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10 — Parameter sweep: mould temperature vs. warp risk

Scan mould temperature from 15 °C to 80 °C and observe how the peak warp-risk score changes.

In [ ]:
T_mold_range = np.linspace(288, 353, 12)  # 15 °C – 80 °C in K
sweep_results = []

for T_mold_K in T_mold_range:
    p                = CoolingParams()
    p.T_init         = params.T_init
    p.T_mold         = float(T_mold_K)
    p.dt             = params.dt
    p.total_time     = params.total_time
    p.avrami_k0      = params.avrami_k0
    p.avrami_n       = params.avrami_n
    p.chi_max        = params.chi_max
    p.warp_temp_coeff = params.warp_temp_coeff
    p.warp_chi_coeff  = params.warp_chi_coeff

    s   = ArtifexCoolingSim(config, device=DEVICE)
    res = s.simulate_cooling(p)
    sweep_results.append({
        "T_mold_C":    T_mold_K - 273.15,
        "max_risk":    res["max_warp_risk"],
        "avg_chi":     res["avg_chi_groove"],
        "is_ok":       res["is_ok"],
    })

T_mold_C = [r["T_mold_C"]  for r in sweep_results]
max_risk  = [r["max_risk"]  for r in sweep_results]
avg_chi   = [r["avg_chi"]   for r in sweep_results]
colors    = ["green" if r["is_ok"] else "red" for r in sweep_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(T_mold_C, max_risk, width=3.5, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(15.0, color="gray", linestyle="--", label="Quality threshold")
ax.set_xlabel("Mould temperature (°C)")
ax.set_ylabel("Max warp-risk score")
ax.set_title("Peak warp risk vs. mould temperature")
ax.legend()

ax = axes[1]
ax.bar(T_mold_C, avg_chi, width=3.5, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(0.15, color="gray", linestyle="--", label="χ threshold")
ax.set_xlabel("Mould temperature (°C)")
ax.set_ylabel("Mean groove crystallinity χ")
ax.set_title("Groove crystallinity vs. mould temperature")
ax.legend()

# Legend patch
from matplotlib.patches import Patch
legend_els = [Patch(facecolor="green", label="Pass"), Patch(facecolor="red", label="Fail")]
for ax in axes:
    ax.legend(handles=ax.get_legend().legend_handles + legend_els)

plt.tight_layout()
plt.show()

## Next steps

| Upgrade | Why it matters |
|---|---|
| **Nakamura kinetics from DSC data** | Current Avrami constants are placeholders; real PET+CSR+PTFE kinetics differ substantially. |
| **Thermoelastic / eigenstrain plate solver** | Convert `warp_risk` from a dimensionless score into predicted out-of-plane deflection (mm). |
| **Separate top/bottom mould BCs** | Real injection tools often have asymmetric cooling channels. |
| **Isaac Sim USD export** | Map the final temperature and crystallinity fields onto a `wp.vec3` mesh for visualisation in Isaac Sim. |
| **Automatic time-step control** | Use an adaptive dt to skip the manual Fourier-stability calculation. |